In [1]:
import json
import torch
import torch.nn as nn
import numpy as np
import os
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import (
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import EarlyStoppingCallback

version = 18

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0601 17:36:10.224000 13008 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[tensorflow|WARNING]From c:\Users\587978\.conda\envs\epu\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# 1. Load base model with Unsloth FastLanguageModel

MODEL_NAME = "unsloth/Mistral-Small-3.1-24B-Instruct-2503"
OUTPUT_DIR = "models/Mistral-Small-3.1-24B-goemotions"# ---------------------------------------------------------------------------
# 1. Load base model with Unsloth FastLanguageModel
# ---------------------------------------------------------------------------
print(f"Loading base model from {MODEL_NAME}")

base_model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)
print("Base model loaded successfully")

Loading base model from unsloth/Mistral-Small-3.1-24B-Instruct-2503
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX 6000 Ada Generation. Num GPUs = 1. Max memory: 47.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base model loaded successfully


In [3]:
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    
)
base_model.print_trainable_parameters()

trainable params: 101,449,728 || all params: 24,112,811,008 || trainable%: 0.4207


In [4]:
# ---------------------------------------------------------------------------
# 3. Data loading — augmented pre-computed embeddings
#    Format: {"X": [[float, ...], ...], "y": [[float, ...], ...]}
#    X[i] = 512-dim sentence embedding, y[i] = 28-dim multi-hot label vector
# ---------------------------------------------------------------------------

BASE = r"C:\Users\587978\Research\Unsloth_model\data_split\augmented3"


def load_split(path: str) -> Dataset:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    return Dataset.from_dict({"input_embeds": d["X"], "labels": d["y"]})


print("Loading augmented splits...")
train_dataset = load_split(f"{BASE}/train.json")
val_dataset   = load_split(f"{BASE}/val.json")
test_dataset  = load_split(f"{BASE}/test.json")

EMBED_DIM = len(train_dataset[0]["input_embeds"])
print(f"Splits loaded — embed_dim: {EMBED_DIM}, "
      f"train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")
print("Dataset formatted successfully")

Loading augmented splits...
Splits loaded — embed_dim: 512, train: 89201, val: 7891, test: 7891
Dataset formatted successfully


In [5]:
# Focal loss constants
EMOTION_LABELS = [    
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NUM_LABELS = len(EMOTION_LABELS)

FOCAL_ALPHA_DEFAULT   = 0.25
FOCAL_ALPHA_PREFERRED = 0.75

PREFERRED_LABELS = {
    "fear", "sadness", "disgust", "disapproval", "annoyance",
    "anger", "disappointment", "optimism", "amusement", "surprise",
    "admiration", "excitement", "confusion","joy","love"
}


FOCAL_ALPHA_PER_LABEL: list[float] = [
    FOCAL_ALPHA_PREFERRED if lbl in PREFERRED_LABELS else FOCAL_ALPHA_DEFAULT
    for lbl in EMOTION_LABELS
]

# Focal loss
class FocalLossWithAlpha(nn.Module):
    """Per-label weighted focal binary cross-entropy for multi-label problems."""
    def __init__(self, alpha: list[float], gamma: float = 2.0):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs   = torch.sigmoid(logits)
        p_t     = probs * targets + (1.0 - probs) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        focal_w = alpha_t * (1.0 - p_t) ** self.gamma
        bce     = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        return (focal_w * bce).mean()


In [6]:
# Classification wrapper

class MistralForMultiLabel(nn.Module):
    is_loaded_in_4bit = True

    def __init__(self, backbone: nn.Module, num_labels: int,
                 hidden_size: int, embed_dim: int):
        super().__init__()
        self.backbone = backbone
        _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.projection = nn.Sequential(
            nn.Linear(embed_dim, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, hidden_size),
        ).to(_device)
        self.dropout    = nn.Dropout(0.1).to(_device)
        self.classifier = nn.Linear(hidden_size, num_labels).to(_device)
        self.focal_loss = FocalLossWithAlpha(FOCAL_ALPHA_PER_LABEL).to(_device)

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs)

    def gradient_checkpointing_disable(self):
        self.backbone.gradient_checkpointing_disable()

    def forward(
        self,
        input_embeds: torch.Tensor,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        B = input_embeds.size(0)
        projected = self.projection(input_embeds).unsqueeze(1)
        attn_mask = torch.ones(B, 1, device=input_embeds.device)

        outputs = self.backbone.base_model.model.model(
            inputs_embeds=projected,
            attention_mask=attn_mask,
            output_hidden_states=True,
        )
        pooled = outputs.hidden_states[-1][:, 0, :]
        logits = self.classifier(self.dropout(pooled))

        loss = self.focal_loss(logits, labels.float()) if labels is not None else None
        return {"loss": loss, "logits": logits}


cfg = base_model.config
hidden_size = (
    getattr(cfg, "hidden_size", None)
    or getattr(getattr(cfg, "text_config", None), "hidden_size", None)
    or base_model.base_model.model.embed_tokens.weight.shape[-1]
)
print(f"hidden_size resolved: {hidden_size}")
model = MistralForMultiLabel(base_model, NUM_LABELS, hidden_size, EMBED_DIM)
print("Classification wrapper created successfully")

hidden_size resolved: 5120
Classification wrapper created successfully


In [7]:
# 4. Custom Trainer with BCE loss + multi-label metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    from sklearn.metrics import accuracy_score

    exact_accuracy  = accuracy_score(labels, preds)
    macro_f1        = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1        = f1_score(labels, preds, average="micro", zero_division=0)
    macro_precision = precision_score(labels, preds, average="macro", zero_division=0)
    macro_recall    = recall_score(labels, preds, average="macro", zero_division=0)

    per_class_f1        = f1_score(labels, preds, average=None, zero_division=0)
    per_class_recall    = recall_score(labels, preds, average=None, zero_division=0)
    per_class_precision = precision_score(labels, preds, average=None, zero_division=0)
    per_class_accuracy  = (preds == labels).mean(axis=0)

    per_class_metrics = {}
    for i, emotion in enumerate(EMOTION_LABELS):
        per_class_metrics[f"f1_{emotion}"]        = float(per_class_f1[i])
        per_class_metrics[f"recall_{emotion}"]    = float(per_class_recall[i])
        per_class_metrics[f"precision_{emotion}"] = float(per_class_precision[i])
        per_class_metrics[f"accuracy_{emotion}"]  = float(per_class_accuracy[i])

    return {
        "exact_accuracy":   exact_accuracy,
        "macro_f1":         macro_f1,
        "micro_f1":         micro_f1,
        "macro_precision":  macro_precision,
        "macro_recall":     macro_recall,
        **per_class_metrics,
    }

In [8]:
# 6. Training arguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # where checkpoints and logs are written
    eval_strategy="epoch",              # run evaluation once per epoch
    save_strategy="epoch",              # save checkpoint once per epoch
    per_device_train_batch_size=8,      # samples per GPU per step
    per_device_eval_batch_size=16,      # larger batch is fine — no gradients
    gradient_accumulation_steps=4,      # effective batch = 8 × 4 = 32
    num_train_epochs=15,                # two full passes — checkpoint saved after each
    learning_rate=1e-4,                 # peak LR after warmup
    bf16=True,                          # bfloat16 mixed precision — faster on Ampere+
    optim="adamw_8bit",                 # 8-bit AdamW — cuts optimizer memory ~4×
    warmup_ratio=0.05,                  # first 5 % of steps ramp LR from 0 to peak
    lr_scheduler_type="cosine",         # cosine decay from peak LR to ~0
    logging_steps=25,                   # print loss/LR to console every 25 steps
    logging_first_step=True,            # also log step 1 to catch early instability
    load_best_model_at_end=True,        # restore best checkpoint after training ends
    metric_for_best_model="macro_f1",   # criterion used to select the best checkpoint
    greater_is_better=True,             # higher macro_f1 = better
    gradient_checkpointing=False,       # disabled — Unsloth handles this internally
    remove_unused_columns=False,        # keep input_embeds column (not a HF default field)
    save_total_limit=15,                # keep both epoch checkpoints on disk
    weight_decay=0.01,                  # L2 regularisation on all trainable parameters
)


In [9]:
class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def _save_checkpoint(self, model, trial, metrics=None):
        super()._save_checkpoint(model, trial)          # no metrics= — not supported in 4.57
        # Build the exact checkpoint subdir the parent just created
        ckpt_dir = os.path.join(
            self.args.output_dir,
            f"checkpoint-{self.state.global_step}",
        )
        os.makedirs(ckpt_dir, exist_ok=True)
        # Save head weights into the checkpoint subdir
        torch.save({
            "projection": model.projection.state_dict(),
            "classifier":  model.classifier.state_dict(),
        }, os.path.join(ckpt_dir, "head_weights.pt"))
        # Save LoRA adapter into the checkpoint subdir
        model.backbone.save_pretrained(os.path.join(ckpt_dir, "lora_adapter"))
        print(f"Checkpoint saved: {ckpt_dir}")

    def _load_best_model(self):
        best_ckpt = self.state.best_model_checkpoint
        if not best_ckpt:
            print("WARNING: no best_model_checkpoint recorded — keeping current weights")
            return
        # Restore head
        head_path = os.path.join(best_ckpt, "head_weights.pt")
        if os.path.exists(head_path):
            head = torch.load(head_path, map_location="cpu")
            self.model.projection.load_state_dict(head["projection"])
            self.model.classifier.load_state_dict(head["classifier"])
            print(f"Head restored from: {best_ckpt}")
        else:
            print(f"WARNING: head_weights.pt not found in {best_ckpt}")
        # Restore LoRA adapter
        lora_path = os.path.join(best_ckpt, "lora_adapter")
        if os.path.exists(lora_path):
            self.model.backbone.load_adapter(lora_path, adapter_name="default")
            print(f"LoRA restored from: {best_ckpt}")
        else:
            print(f"WARNING: lora_adapter/ not found in {best_ckpt}")

In [10]:
trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 89,201 | Num Epochs = 15 | Total steps = 41,820
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 116,018,716 of 13,245,258,268 (0.88% trained)


Epoch,Training Loss,Validation Loss,Exact Accuracy,Macro F1,Micro F1,Macro Precision,Macro Recall,F1 Admiration,Recall Admiration,Precision Admiration,Accuracy Admiration,F1 Amusement,Recall Amusement,Precision Amusement,Accuracy Amusement,F1 Anger,Recall Anger,Precision Anger,Accuracy Anger,F1 Annoyance,Recall Annoyance,Precision Annoyance,Accuracy Annoyance,F1 Approval,Recall Approval,Precision Approval,Accuracy Approval,F1 Caring,Recall Caring,Precision Caring,Accuracy Caring,F1 Confusion,Recall Confusion,Precision Confusion,Accuracy Confusion,F1 Curiosity,Recall Curiosity,Precision Curiosity,Accuracy Curiosity,F1 Desire,Recall Desire,Precision Desire,Accuracy Desire,F1 Disappointment,Recall Disappointment,Precision Disappointment,Accuracy Disappointment,F1 Disapproval,Recall Disapproval,Precision Disapproval,Accuracy Disapproval,F1 Disgust,Recall Disgust,Precision Disgust,Accuracy Disgust,F1 Embarrassment,Recall Embarrassment,Precision Embarrassment,Accuracy Embarrassment,F1 Excitement,Recall Excitement,Precision Excitement,Accuracy Excitement,F1 Fear,Recall Fear,Precision Fear,Accuracy Fear,F1 Gratitude,Recall Gratitude,Precision Gratitude,Accuracy Gratitude,F1 Grief,Recall Grief,Precision Grief,Accuracy Grief,F1 Joy,Recall Joy,Precision Joy,Accuracy Joy,F1 Love,Recall Love,Precision Love,Accuracy Love,F1 Nervousness,Recall Nervousness,Precision Nervousness,Accuracy Nervousness,F1 Optimism,Recall Optimism,Precision Optimism,Accuracy Optimism,F1 Pride,Recall Pride,Precision Pride,Accuracy Pride,F1 Realization,Recall Realization,Precision Realization,Accuracy Realization,F1 Relief,Recall Relief,Precision Relief,Accuracy Relief,F1 Remorse,Recall Remorse,Precision Remorse,Accuracy Remorse,F1 Sadness,Recall Sadness,Precision Sadness,Accuracy Sadness,F1 Surprise,Recall Surprise,Precision Surprise,Accuracy Surprise,F1 Neutral,Recall Neutral,Precision Neutral,Accuracy Neutral
1,0.015800,0.014762,0.212267,0.388770,0.394169,0.522567,0.403089,0.548730,0.593407,0.510309,0.876188,0.579358,0.575862,0.582897,0.938538,0.243070,0.146907,0.703704,0.955012,0.343299,0.555927,0.248322,0.838550,0.032544,0.016692,0.647059,0.917121,0.153333,0.086466,0.676471,0.967811,0.305750,0.353896,0.269136,0.937270,0.168190,0.094650,0.754098,0.942339,0.254335,0.160584,0.611111,0.983652,0.105263,0.059859,0.435897,0.963376,0.349810,0.370221,0.331532,0.913319,0.316327,0.629442,0.211244,0.932075,0.620690,0.521739,0.765957,0.994424,0.296925,0.660377,0.191518,0.915980,0.641509,0.555556,0.758929,0.987961,0.782979,0.658083,0.966387,0.967685,0.461538,0.300000,1.000000,0.998226,0.350598,0.253602,0.567742,0.958687,0.597518,0.705021,0.518462,0.942466,0.724638,0.657895,0.806452,0.997592,0.378633,0.614796,0.273553,0.899759,0.800000,0.923077,0.705882,0.998479,0.000000,0.000000,0.000000,0.964517,0.676923,0.733333,0.628571,0.997339,0.383721,0.266129,0.687500,0.986567,0.444444,0.398693,0.502058,0.961348,0.325444,0.394265,0.277078,0.942213,0.000000,0.000000,0.000000,0.925738
2,0.011500,0.012971,0.271448,0.494574,0.452550,0.659507,0.516404,0.570098,0.550450,0.591202,0.894690,0.586842,0.768966,0.474468,0.920416,0.438125,0.734536,0.312158,0.907363,0.170564,0.103506,0.484375,0.923584,0.061493,0.031866,0.875000,0.918768,0.178808,0.101504,0.750000,0.968572,0.398020,0.652597,0.286325,0.922950,0.133333,0.072016,0.897436,0.942339,0.549763,0.423358,0.783784,0.987961,0.371429,0.411972,0.338150,0.949816,0.207455,0.128773,0.533333,0.938031,0.438519,0.751269,0.309623,0.951971,0.785714,0.797101,0.774648,0.996198,0.530612,0.490566,0.577778,0.976682,0.761329,0.823529,0.707865,0.989989,0.768154,0.628040,0.988739,0.966417,0.857143,0.750000,1.000000,0.999366,0.442786,0.512968,0.389497,0.943226,0.625000,0.564854,0.699482,0.958941,0.972973,0.947368,1.000000,0.999747,0.506460,0.500000,0.513089,0.951590,0.862745,0.846154,0.880000,0.999113,0.061856,0.032143,0.818182,0.965404,0.885246,0.900000,0.870968,0.999113,0.704225,0.604839,0.842697,0.992016,0.507212,0.689542,0.401141,0.948042,0.465359,0.637993,0.

Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-2788
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-5576
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-8364
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-11152
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-13940
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-16728
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-19516
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-22304
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-25092
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-27880
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-30668
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-33456
Checkpoint saved: models/Mistral-Small-3.1-24B-goemotions\checkpoint-36244
Checkpoint saved: models/Mis

TrainOutput(global_step=41820, training_loss=0.004811599533451818, metrics={'train_runtime': 64612.5284, 'train_samples_per_second': 20.708, 'train_steps_per_second': 0.647, 'total_flos': 0.0, 'train_loss': 0.004811599533451818, 'epoch': 15.0})

In [11]:
# ── Save everything needed for local inference on raw text ─────────────────
import json

_model_slug = f"Mistral-Small-3.1-24B-goemotions_v{version}"
_save_dir   = os.path.join(r"C:\Users\587978\Research\Unsloth_model\models", _model_slug)
os.makedirs(_save_dir, exist_ok=True)

# 1. LoRA adapter
_lora_dir = os.path.join(_save_dir, "lora_adapter")
model.backbone.save_pretrained(_lora_dir)
print(f"LoRA adapter  : {_lora_dir}")

# 2. Projection + classifier head
_head_path = os.path.join(_save_dir, "head_weights.pt")
torch.save({
    "projection": model.projection.state_dict(),
    "classifier":  model.classifier.state_dict(),
}, _head_path)
print(f"Head weights  : {_head_path}")

# 3. Architecture + encoder config
_config = {
    "model_name":     MODEL_NAME,
    "lora_rank":      16,
    "lora_alpha":     32,
    "lora_dropout":   0.0,
    "hidden_size":    hidden_size,
    "embed_dim":      EMBED_DIM,
    "num_labels":     NUM_LABELS,
    "emotion_labels": EMOTION_LABELS,
    "encoder_type":   "tfhub",
    "encoder_name":   "https://tfhub.dev/google/universal-sentence-encoder/4",
    "threshold":      0.5,
}
with open(os.path.join(_save_dir, "config.json"), "w") as f:
    json.dump(_config, f, indent=2)
print(f"config.json   : {_save_dir}")

# 4. Focal loss constants
_focal = {
    "preferred_labels":      list(PREFERRED_LABELS),
    "focal_alpha_per_label": FOCAL_ALPHA_PER_LABEL,
    "focal_alpha_default":   FOCAL_ALPHA_DEFAULT,
    "focal_alpha_preferred": FOCAL_ALPHA_PREFERRED,
}
with open(os.path.join(_save_dir, "focal_config.json"), "w") as f:
    json.dump(_focal, f, indent=2)

# 5. Self-contained inference helper script
_infer_script = """
\"\"\"
Standalone inference on raw text.
Usage:
    from infer import EmotionClassifier
    clf = EmotionClassifier(r"<path_to_model_dir>")
    print(clf.predict(["I am so happy today!", "This makes me angry."]))
\"\"\"
import json, os, torch, torch.nn as nn, numpy as np
import tensorflow_hub as hub
from unsloth import FastLanguageModel

class FocalLossWithAlpha(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p_t   = probs*targets + (1-probs)*(1-targets)
        a_t   = self.alpha*targets + (1-self.alpha)*(1-targets)
        bce   = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        return (a_t*(1-p_t)**self.gamma * bce).mean()

class MistralForMultiLabel(nn.Module):
    is_loaded_in_4bit = True
    def __init__(self, backbone, num_labels, hidden_size, embed_dim, focal_alpha):
        super().__init__()
        self.backbone   = backbone
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.projection = nn.Sequential(
            nn.Linear(embed_dim, hidden_size//2), nn.GELU(),
            nn.Linear(hidden_size//2, hidden_size),
        ).to(dev)
        self.dropout    = nn.Dropout(0.1).to(dev)
        self.classifier = nn.Linear(hidden_size, num_labels).to(dev)
        self.focal_loss = FocalLossWithAlpha(focal_alpha).to(dev)
    def forward(self, input_embeds, labels=None, **kwargs):
        B = input_embeds.size(0)
        projected = self.projection(input_embeds).unsqueeze(1)
        attn_mask = torch.ones(B, 1, device=input_embeds.device)
        outputs   = self.backbone.base_model.model.model(
            inputs_embeds=projected, attention_mask=attn_mask, output_hidden_states=True)
        logits = self.classifier(self.dropout(outputs.hidden_states[-1][:,0,:]))
        loss   = self.focal_loss(logits, labels.float()) if labels is not None else None
        return {"loss": loss, "logits": logits}

class EmotionClassifier:
    def __init__(self, model_dir: str):
        with open(f"{model_dir}/config.json") as f:
            cfg = json.load(f)
        with open(f"{model_dir}/focal_config.json") as f:
            fcfg = json.load(f)
        self.labels    = cfg["emotion_labels"]
        self.threshold = cfg.get("threshold", 0.5)
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        os.environ.setdefault("TFHUB_CACHE_DIR",
            "C:\\\\Users\\\\587978\\\\Research\\\\tfhub_cache")
        self.encoder = hub.load(cfg["encoder_name"])
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=cfg["model_name"], max_seq_length=2,
            load_in_4bit=True, dtype=torch.bfloat16)
        base_model = FastLanguageModel.get_peft_model(
            base_model, r=cfg["lora_rank"], lora_alpha=cfg["lora_alpha"],
            lora_dropout=cfg["lora_dropout"], bias="none",
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            use_gradient_checkpointing="unsloth")
        self.model = MistralForMultiLabel(
            base_model, cfg["num_labels"], cfg["hidden_size"],
            cfg["embed_dim"], fcfg["focal_alpha_per_label"])
        head = torch.load(f"{model_dir}/head_weights.pt", map_location="cpu")
        self.model.projection.load_state_dict(head["projection"])
        self.model.classifier.load_state_dict(head["classifier"])
        lora_path = f"{model_dir}/lora_adapter"
        if os.path.exists(lora_path):
            self.model.backbone.load_adapter(lora_path, adapter_name="default")
        self.model.eval().to(self.device)

    def predict(self, texts: list, threshold: float = None) -> list:
        thr = threshold or self.threshold
        embs = self.encoder(texts).numpy()
        embs = torch.tensor(embs, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model(embs)["logits"].cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))
        return [
            {self.labels[j]: round(float(probs[i,j]),4)
             for j in range(len(self.labels)) if probs[i,j] >= thr}
            for i in range(len(texts))
        ]
"""
_infer_path = os.path.join(_save_dir, "infer.py")
with open(_infer_path, "w", encoding="utf-8") as f:
    f.write(_infer_script)

print(f"\nAll artifacts saved to: {_save_dir}")
print("  lora_adapter/     <- LoRA delta weights")
print("  head_weights.pt   <- projection + classifier")
print("  config.json       <- architecture + encoder")
print("  focal_config.json <- focal loss alphas")
print("  infer.py          <- self-contained inference class")
print(f"\nUsage:\n  from infer import EmotionClassifier")
print(f"  clf = EmotionClassifier(r\"{_save_dir}\")")
print(f"  clf.predict([\"I am so happy!\", \"This is terrible.\"])")

LoRA adapter  : C:\Users\587978\Research\Unsloth_model\models\Mistral-Small-3.1-24B-goemotions_v18\lora_adapter
Head weights  : C:\Users\587978\Research\Unsloth_model\models\Mistral-Small-3.1-24B-goemotions_v18\head_weights.pt
config.json   : C:\Users\587978\Research\Unsloth_model\models\Mistral-Small-3.1-24B-goemotions_v18

All artifacts saved to: C:\Users\587978\Research\Unsloth_model\models\Mistral-Small-3.1-24B-goemotions_v18
  lora_adapter/     <- LoRA delta weights
  head_weights.pt   <- projection + classifier
  config.json       <- architecture + encoder
  focal_config.json <- focal loss alphas
  infer.py          <- self-contained inference class

Usage:
  from infer import EmotionClassifier
  clf = EmotionClassifier(r"C:\Users\587978\Research\Unsloth_model\models\Mistral-Small-3.1-24B-goemotions_v18")
  clf.predict(["I am so happy!", "This is terrible."])


In [12]:
# Performnace - ENG

import os
import numpy as np
import pandas as pd

test_results = trainer.evaluate(test_dataset)
results = {k.replace("eval_", ""): v for k, v in test_results.items()}

# ── Support: count of positive instances per label in test set ─────────────
test_labels = np.array(test_dataset["labels"])
support_per_label = test_labels.sum(axis=0).astype(int)

# ── Create versioned directory ─────────────────────────────────────────────
OUTPUT_DIR    = os.path.join(r"C:\Users\587978\Research\models\performance", _model_slug)
versioned_dir = os.path.join(OUTPUT_DIR, f"v{version}")
os.makedirs(versioned_dir, exist_ok=True)

# ── Per-emotion table ──────────────────────────────────────────────────────
per_emotion_rows = []
for i, emotion in enumerate(EMOTION_LABELS):
    per_emotion_rows.append({
        "emotion":   emotion,
        "accuracy":  round(results.get(f"accuracy_{emotion}",  float("nan")), 4),
        "precision": round(results.get(f"precision_{emotion}", float("nan")), 4),
        "recall":    round(results.get(f"recall_{emotion}",    float("nan")), 4),
        "f1":        round(results.get(f"f1_{emotion}",        float("nan")), 4),
        "support":   int(support_per_label[i]),
    })
per_emotion_df = pd.DataFrame(per_emotion_rows)

# ── Aggregate metrics ──────────────────────────────────────────────────────
aggregate_keys = ["exact_accuracy", "macro_f1", "micro_f1", "macro_precision", "macro_recall"]
aggregate_df = pd.DataFrame([
    {"metric": k, "value": round(results[k], 4)}
    for k in aggregate_keys if k in results
])

# ── Write to versioned xlsx ────────────────────────────────────────────────
model_version = os.path.basename(versioned_dir)
results_xlsx  = os.path.join(versioned_dir, f"test_results_{model_version}.xlsx")
with pd.ExcelWriter(results_xlsx, engine="openpyxl") as writer:
    per_emotion_df.to_excel(writer, sheet_name="per_emotion", index=False)
    aggregate_df.to_excel(writer, sheet_name="aggregate",    index=False)
print(f"Test results saved to: {results_xlsx}")

Test results saved to: C:\Users\587978\Research\models\performance\Mistral-Small-3.1-24B-goemotions_v18\v18\test_results_v18.xlsx


In [13]:
# Print emotions with F1 > 0.5 on test set
results = {k.replace("eval_", ""): v for k, v in test_results.items()}

print("\n=== Emotions with F1 > 0.6 ===")
high_f1 = [
    (emotion, round(results[f"f1_{emotion}"], 5))
    for emotion in EMOTION_LABELS
    if f"f1_{emotion}" in results and results[f"f1_{emotion}"] > 0.5
]
if high_f1:
    for emotion, score in sorted(high_f1, key=lambda x: x[1], reverse=True):
        print(f"  {emotion:<20} {score:.4f}")
else:
    print("  None above 0.6")


=== Emotions with F1 > 0.6 ===
  relief               1.0000
  embarrassment        0.9877
  nervousness          0.9773
  pride                0.9677
  fear                 0.9390
  remorse              0.9353
  desire               0.9288
  grief                0.8980
  caring               0.8942
  disgust              0.8856
  realization          0.8812
  gratitude            0.8804
  excitement           0.8767
  sadness              0.8713
  surprise             0.8505
  disappointment       0.8473
  confusion            0.7899
  optimism             0.7882
  joy                  0.7794
  love                 0.7656
  amusement            0.7611
  anger                0.7395
  curiosity            0.7319
  admiration           0.6844
  disapproval          0.6453
  annoyance            0.6148


In [ ]:
# Performance - IT

country = "IT"

# ── Load test set ──────────────────────────────────────────────────────────
IT_BASE = r"C:\Users\587978\Research\Unsloth_model\data_split\augmented_IT"

def load_split(path: str) -> Dataset:
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    return Dataset.from_dict({"input_embeds": d["X"], "labels": d["y"]})

test_it = load_split(f"{IT_BASE}/test.json")
test_it.set_format("torch")
print(f"Test set [{country}] loaded: {len(test_it)} samples, embed_dim: {len(test_it[0]['input_embeds'])}")

# ── Run evaluation ─────────────────────────────────────────────────────────
it_results_raw = trainer.evaluate(test_it)
results_it = {k.replace("eval_", ""): v for k, v in it_results_raw.items()}

# ── Support ────────────────────────────────────────────────────────────────
it_labels = np.array(test_it["labels"])
support_per_label = it_labels.sum(axis=0).astype(int)

# ── Per-emotion table ──────────────────────────────────────────────────────
per_emotion_rows = []
for i, emotion in enumerate(EMOTION_LABELS):
    per_emotion_rows.append({
        "emotion":   emotion,
        "accuracy":  round(results_it.get(f"accuracy_{emotion}",  float("nan")), 4),
        "precision": round(results_it.get(f"precision_{emotion}", float("nan")), 4),
        "recall":    round(results_it.get(f"recall_{emotion}",    float("nan")), 4),
        "f1":        round(results_it.get(f"f1_{emotion}",        float("nan")), 4),
        "support":   int(support_per_label[i]),
    })
per_emotion_df = pd.DataFrame(per_emotion_rows)

# ── Aggregate metrics ──────────────────────────────────────────────────────
aggregate_keys = ["exact_accuracy", "macro_f1", "micro_f1", "macro_precision", "macro_recall"]
aggregate_df = pd.DataFrame([
    {"metric": k, "value": round(results_it[k], 4)}
    for k in aggregate_keys if k in results_it
])

# ── Save results ───────────────────────────────────────────────────────────
versioned_dir = r"C:\Users\587978\Research\models\performance\Mistral-Small-3.1-24B-goemotions_v13\v13"
os.makedirs(versioned_dir, exist_ok=True)

xlsx = os.path.join(versioned_dir, f"test_results_v{version}_{country}.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    per_emotion_df.to_excel(writer, sheet_name="per_emotion", index=False)
    aggregate_df.to_excel(writer, sheet_name="aggregate",    index=False)
print(f"Results saved to: {xlsx}")

print(f"\n=== Emotions with F1 > 0.5 ===")
high_f1 = [
    (emotion, round(results_it[f"f1_{emotion}"], 5))
    for emotion in EMOTION_LABELS
    if f"f1_{emotion}" in results_it and results_it[f"f1_{emotion}"] > 0.5
]
for e, s in sorted(high_f1, key=lambda x: x[1], reverse=True):
    print(f"  {e:<20} {s:.4f}")